<a href="https://colab.research.google.com/github/kabila96/-Crop-Recommendation-System/blob/main/Prompt_Refiner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# PROMPT REFINEMENT ENGINE
# Google Colab + Gradio + OpenAI Responses API
# ============================================================

# 1. Install the required libraries
!pip install -q --upgrade openai gradio

# 2. Import the required libraries
import os
from getpass import getpass

import gradio as gr
from openai import OpenAI


# ============================================================
# 3. SET THE OPENAI API KEY AS AN ENVIRONMENT VARIABLE
# ============================================================

# Your key is entered privately and will not be displayed in Colab.
# Create an API key at:
# https://platform.openai.com/api-keys

if not os.environ.get("OPENAI_API_KEY"):
    api_key = getpass("Enter your OpenAI API key: ").strip()

    if not api_key:
        raise ValueError("No API key was provided.")

    os.environ["OPENAI_API_KEY"] = api_key

print("OpenAI API key loaded successfully.")


# ============================================================
# 4. CREATE THE OPENAI CLIENT
# ============================================================

client = OpenAI()

# Change this model if necessary.
# gpt-5.6-luna is suitable for efficient, high-volume tasks.
MODEL_NAME = "gpt-5.6-luna"


# ============================================================
# 5. SYSTEM PROMPT FOR THE REFINEMENT ENGINE
# ============================================================

SYSTEM_PROMPT = """
You are an expert Prompt Refinement Engine.

Your job is to transform a user's rough, vague, incomplete, disorganised,
or messy prompt into a precise, structured and effective prompt that can
be copied and used with an AI assistant.

Preserve the user's original intention. Do not change the central objective.

Improve the prompt by:

1. Identifying the most appropriate role for the AI.
2. Clearly defining the task.
3. Adding useful context and relevant requirements.
4. Specifying a suitable output format.
5. Defining the appropriate tone and writing style.
6. Expanding important details that the user may have omitted.
7. Identifying assumptions, constraints and success criteria where relevant.
8. Removing repetition, ambiguity and conflicting instructions.
9. Using placeholders such as [target audience] or [word count] when
   essential information cannot safely be inferred.
10. Producing a final standalone prompt that can be copied immediately.

Do not answer or perform the user's original task. Only refine the prompt.

Return the result using exactly the following Markdown structure:

## 1. Role
State the expert role that the AI should adopt.

## 2. Task
Explain the main task clearly and precisely.

## 3. Context and Requirements
List the important background information, scope, constraints and requirements.

## 4. Output Format
Specify how the answer should be organised and presented.

## 5. Tone
Specify the appropriate tone, style and level of technical detail.

## 6. Additional Detail Expansion
Add useful details, assumptions, quality criteria or placeholders that would
make the prompt more reliable. Clearly label any assumptions.

## 7. Complete Refined Prompt
Provide one complete, standalone and copy-ready prompt inside a Markdown
code block.

The Complete Refined Prompt must include all important instructions from
the previous sections. It must not refer to phrases such as "the prompt above"
or "the information provided earlier".
"""


# ============================================================
# 6. PROMPT REFINEMENT FUNCTION
# ============================================================

def refine_prompt(messy_prompt):
    """
    Sends a rough prompt to the OpenAI API and returns a structured,
    copy-ready version.
    """

    if messy_prompt is None or not messy_prompt.strip():
        return "Please enter a prompt before clicking **Refine Prompt**."

    cleaned_prompt = messy_prompt.strip()

    # Limit extremely large submissions.
    if len(cleaned_prompt) > 20_000:
        return (
            "The prompt is too long. Please reduce it to fewer than "
            "20,000 characters."
        )

    try:
        response = client.responses.create(
            model=MODEL_NAME,
            instructions=SYSTEM_PROMPT,
            input=(
                "Refine the following messy prompt. Preserve its intended "
                "meaning and return the required seven-section structure.\n\n"
                f"MESSY PROMPT:\n{cleaned_prompt}"
            ),
            reasoning={"effort": "low"},
            text={"verbosity": "medium"}
        )

        refined_result = response.output_text

        if not refined_result:
            return "The model returned an empty response. Please try again."

        return refined_result

    except Exception as error:
        error_name = type(error).__name__

        return f"""
### The prompt could not be refined

**Error type:** `{error_name}`

**Details:** {str(error)}

Check that:

- your API key is correct;
- your OpenAI API account has available credit;
- the selected model is available to your account; and
- the Colab runtime has an internet connection.
"""


# ============================================================
# 7. BUILD THE GRADIO INTERFACE
# ============================================================

with gr.Blocks(theme=gr.themes.Soft(), title="Prompt Refinement Engine") as demo:

    gr.Markdown(
        """
        # Prompt Refinement Engine

        Transform a rough or disorganised idea into a clear, structured and
        copy-ready AI prompt.

        Enter your draft below and select **Refine Prompt**. The result will
        contain Role, Task, Context and Requirements, Output Format, Tone,
        Additional Detail Expansion and a Complete Refined Prompt.
        """
    )

    with gr.Row():

        with gr.Column(scale=1):
            messy_prompt_input = gr.Textbox(
                label="Messy Prompt",
                placeholder=(
                    "Example: Write something about climate change in cities "
                    "and include data and recommendations."
                ),
                lines=16
            )

            with gr.Row():
                refine_button = gr.Button(
                    "Refine Prompt",
                    variant="primary"
                )

                clear_button = gr.ClearButton(
                    components=[],
                    value="Clear"
                )

        with gr.Column(scale=1):
            refined_prompt_output = gr.Markdown(
                label="Refined Prompt",
                value=(
                    "Your structured prompt will appear here after you select "
                    "**Refine Prompt**."
                )
            )

    # Connect the Clear button after all components exist.
    clear_button.add(
        components=[messy_prompt_input, refined_prompt_output]
    )

    # Run the refinement function when the button is selected.
    refine_button.click(
        fn=refine_prompt,
        inputs=messy_prompt_input,
        outputs=refined_prompt_output
    )

    # Allow Ctrl+Enter to submit from the textbox.
    messy_prompt_input.submit(
        fn=refine_prompt,
        inputs=messy_prompt_input,
        outputs=refined_prompt_output
    )

    gr.Markdown("## Example prompts")

    gr.Examples(
        examples=[
            [
                "Write a report about urban heat in London and Birmingham "
                "using satellite data."
            ],
            [
                "Help me make a CV for a GIS job. I know Python, QGIS and "
                "ArcGIS and recently completed a master's degree."
            ],
            [
                "Analyse my supermarket sales data and show the important "
                "results with graphs."
            ],
            [
                "Create a business plan for a small mobile kitchen in "
                "Zimbabwe that accepts USD, EcoCash and South African rand."
            ],
            [
                "Explain climate change to students and give examples from "
                "Southern Africa."
            ]
        ],
        inputs=messy_prompt_input
    )

    gr.Markdown(
        """
        **Security note:** Never paste your OpenAI API key into the prompt box
        or expose it in a public notebook. API usage may incur charges through
        your OpenAI API account.
        """
    )


# ============================================================
# 8. LAUNCH THE APPLICATION
# ============================================================

# share=True creates a temporary public Gradio link for the Colab session.
# Stop the Colab cell when you want to close the application.

demo.launch(
    share=True,
    debug=True
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.4/94.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.0/83.0 kB 3.1 MB/s eta 0:00:00
Enter your OpenAI API key: ··········
OpenAI API key loaded successfully.


/tmp/ipykernel_5618/3040999800.py:172: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Prompt Refinement Engine") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://135691db9abc632bf9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://135691db9abc632bf9.gradio.live
